In [7]:
import pandas as pd
import numpy as np

PORT_HEDLAND_LAT = -20.3
PORT_HEDLAND_LON = 118.6

# IBTrACS often has a units row right after the header — skip it
df = pd.read_csv('/Users/kdpaa/Documents/Pilbara/pilbara-disruption-forecaster/data/raw/ibtracs_southern_hemisphere.csv', skiprows=[1], low_memory=False)

print(df.shape)
print(df.columns.tolist())
df.head()

(170230, 174)
['SID', 'SEASON', 'NUMBER', 'BASIN', 'SUBBASIN', 'NAME', 'ISO_TIME', 'NATURE', 'LAT', 'LON', 'WMO_WIND', 'WMO_PRES', 'WMO_AGENCY', 'TRACK_TYPE', 'DIST2LAND', 'LANDFALL', 'IFLAG', 'USA_AGENCY', 'USA_ATCF_ID', 'USA_LAT', 'USA_LON', 'USA_RECORD', 'USA_STATUS', 'USA_WIND', 'USA_PRES', 'USA_SSHS', 'USA_R34_NE', 'USA_R34_SE', 'USA_R34_SW', 'USA_R34_NW', 'USA_R50_NE', 'USA_R50_SE', 'USA_R50_SW', 'USA_R50_NW', 'USA_R64_NE', 'USA_R64_SE', 'USA_R64_SW', 'USA_R64_NW', 'USA_POCI', 'USA_ROCI', 'USA_RMW', 'USA_EYE', 'TOKYO_LAT', 'TOKYO_LON', 'TOKYO_GRADE', 'TOKYO_WIND', 'TOKYO_PRES', 'TOKYO_R50_DIR', 'TOKYO_R50_LONG', 'TOKYO_R50_SHORT', 'TOKYO_R30_DIR', 'TOKYO_R30_LONG', 'TOKYO_R30_SHORT', 'TOKYO_LAND', 'CMA_LAT', 'CMA_LON', 'CMA_CAT', 'CMA_WIND', 'CMA_PRES', 'HKO_LAT', 'HKO_LON', 'HKO_CAT', 'HKO_WIND', 'HKO_PRES', 'KMA_LAT', 'KMA_LON', 'KMA_CAT', 'KMA_WIND', 'KMA_PRES', 'KMA_R50_DIR', 'KMA_R50_LONG', 'KMA_R50_SHORT', 'KMA_R30_DIR', 'KMA_R30_LONG', 'KMA_R30_SHORT', 'NEWDELHI_LAT', 'NEW

,SID,SEASON,NUMBER,BASIN,SUBBASIN,NAME,ISO_TIME,NATURE,LAT,LON,...,BOM_GUST_PER,REUNION_GUST,REUNION_GUST_PER,USA_SEAHGT,USA_SEARAD_NE,USA_SEARAD_SE,USA_SEARAD_SW,USA_SEARAD_NW,STORM_SPEED,STORM_DIR
0,1848011S09079,1848,1,SI,MM,UNNAMED,1848-01-11 06:00:00,NR,-9.0,79.0,...,,,,,,,,,26,250
1,1848011S09079,1848,1,SI,MM,UNNAMED,1848-01-11 09:00:00,NR,-9.4,77.7,...,,,,,,,,,26,250
2,1848011S09079,1848,1,SI,MM,UNNAMED,1848-01-11 12:00:00,NR,-9.8,76.5,...,,,,,,,,,26,250
3,1848011S09079,1848,1,SI,MM,UNNAMED,1848-01-11 15:00:00,NR,-10.3,75.2,...,,,,,,,,,26,250
4,1848011S09079,1848,1,SI,MM,UNNAMED,1848-01-11 18:00:00,NR,-10.8,74.0,...,,,,,,,,,26,245


In [8]:
cols = ['SID', 'NAME', 'ISO_TIME', 'LAT', 'LON', 'USA_WIND', 'USA_PRES']
df = df[cols].copy()

df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
df['USA_WIND'] = pd.to_numeric(df['USA_WIND'], errors='coerce')
df['ISO_TIME'] = pd.to_datetime(df['ISO_TIME'], errors='coerce')

df = df.dropna(subset=['LAT', 'LON', 'ISO_TIME'])

df['dist_km'] = np.sqrt(
    (df['LAT'] - PORT_HEDLAND_LAT)**2 +
    (df['LON'] - PORT_HEDLAND_LON)**2
) * 111

pilbara_cyclones = df[df['dist_km'] <= 500].copy()
print(f"Found {pilbara_cyclones['SID'].nunique()} unique storms near Pilbara")

Found 336 unique storms near Pilbara


In [9]:
def wind_to_australian_category(wind_knots):
    """Convert max sustained wind (knots) to Australian TC category (1-5)."""
    wind_kmh = wind_knots * 1.852
    if wind_kmh < 63:
        return 0
    elif wind_kmh < 88:
        return 1
    elif wind_kmh < 117:
        return 2
    elif wind_kmh < 159:
        return 3
    elif wind_kmh < 200:
        return 4
    else:
        return 5

pilbara_cyclones['category'] = pilbara_cyclones['USA_WIND'].apply(wind_to_australian_category)
pilbara_cyclones['category'].value_counts()

category
5    4300
0     710
1     663
2     567
3     563
4     262
Name: count, dtype: int64

In [10]:
print(pilbara_cyclones['ISO_TIME'].min(), "to", pilbara_cyclones['ISO_TIME'].max())

veronica = pilbara_cyclones[pilbara_cyclones['NAME'].str.contains('VERONICA', case=False, na=False)]
print(veronica[['NAME','ISO_TIME','category','dist_km']])

zelia = pilbara_cyclones[pilbara_cyclones['NAME'].str.contains('ZELIA', case=False, na=False)]
print(zelia[['NAME','ISO_TIME','category','dist_km']])

1897-01-26 12:00:00 to 2026-03-26 15:00:00
            NAME            ISO_TIME  category     dist_km
159005  VERONICA 2019-03-20 18:00:00         5  498.512357
159006  VERONICA 2019-03-20 21:00:00         5  476.783441
159007  VERONICA 2019-03-21 00:00:00         5  444.277413
159008  VERONICA 2019-03-21 03:00:00         5  425.435647
159009  VERONICA 2019-03-21 06:00:00         5  407.235386
159010  VERONICA 2019-03-21 09:00:00         5  379.353766
159011  VERONICA 2019-03-21 12:00:00         5  362.920776
159012  VERONICA 2019-03-21 15:00:00         5  357.447577
159013  VERONICA 2019-03-21 18:00:00         5  362.411175
159014  VERONICA 2019-03-21 21:00:00         5  357.964244
159015  VERONICA 2019-03-22 00:00:00         5  354.157846
159016  VERONICA 2019-03-22 03:00:00         5  354.157846
159017  VERONICA 2019-03-22 06:00:00         4  344.815385
159018  VERONICA 2019-03-22 09:00:00         4  335.580005
159019  VERONICA 2019-03-22 12:00:00         4  310.998151
159020  VERON

In [11]:
pilbara_cyclones.to_csv('../data/processed/pilbara_cyclones.csv', index=False)
print("Saved!")

Saved!


In [12]:
all_recent = pilbara_cyclones[pilbara_cyclones['ISO_TIME'] >= '2025-01-01']
print(all_recent[['NAME','ISO_TIME','category','dist_km']].drop_duplicates(subset='NAME'))

            NAME            ISO_TIME  category     dist_km
167883      SEAN 2025-01-17 00:00:00         5  492.919994
168380     ZELIA 2025-02-11 00:00:00         5  474.192208
169875  MITCHELL 2026-02-06 00:00:00         0  459.545362
170168   NARELLE 2026-03-24 18:00:00         2  487.642595
